# Yahoo Finance ('yfinance') Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [1]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [ ]:
# Let's create a list of sectors
sector_etfs = ["SPY","QQQ","XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY"]

prices = yf.download(sector_etfs, period='max', auto_adjust=True)['Close']

[*********************100%***********************]  11 of 11 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [3]:
prices.dropna(inplace=True)
prices.head()



Price           Close                                                       \
Ticker            QQQ        SPY       XLB       XLE        XLF        XLI   
Date                                                                         
1999-03-10  43.128662  80.514320  6.032154  6.018414  12.424142  15.783932   
1999-03-11  43.339794  81.410233  6.096782  6.103726  12.484308  15.803361   
1999-03-12  42.284035  80.631149  6.070932  6.080459  12.506873  15.851918   
1999-03-15  43.498180  81.780251  6.083858  6.033926  12.544479  15.813079   
1999-03-16  43.867702  81.468620  6.058005  6.002908  12.363979  15.706223   

Price                                                  ...   Volume           \
Ticker            XLK        XLP       XLU        XLV  ...      SPY      XLB   
Date                                                   ...                     
1999-03-10  13.154646  14.296957  5.565430  19.005713  ...  3950000  53600.0   
1999-03-11  13.131359  14.437928  5.608499  19.131994  ...  6583700  27800.0   
1999-03-12  13.014944  14.562325  5.670029  19.026760  ...  5286500  15800.0   
1999-03-15  13.224493  14.686716  5.685412  19.089911  ...  5394400  13400.0   
1999-03-16  13.451498  14.753060  5.663879  19.047806  ...  4547500  15000.0   

Price                                                                  \
Ticker            XLE       XLF     XLI        XLK       XLP      XLU   
Date                                                                    
1999-03-10  3467600.0   91217.0  2700.0   884000.0   32600.0  68000.0   
1999-03-11  1818000.0  210993.0  1800.0  2432400.0   55700.0  19200.0   
1999-03-12  1208600.0  183911.0  2200.0  1672000.0   48100.0  15600.0   
1999-03-15  1046000.0   82969.0  1900.0   864200.0  246400.0  31800.0   
1999-03-16   546600.0  188466.0  4800.0  1517600.0   63700.0  20200.0   

Price                          
Ticker           XLV      XLY  
Date                           
1999-03-10   10700.0  11200.0  
1999-03-11   23600.0  26200.0  
1999-03-12   24600.0  27400.0  
1999-03-15  144800.0  14200.0  
1999-03-16    9100.0  12400.0  

[5 rows x 22 columns]

In [4]:
prices.tail()

Price            Close                                               \
Ticker             QQQ         SPY        XLB        XLE        XLF   
Date                                                                  
2026-02-23  601.409973  682.390015  52.959999  55.150002  50.730000   
2026-02-24  607.869995  687.349976  53.360001  55.099998  50.980000   
2026-02-25  616.679993  693.150024  53.060001  54.869999  51.869999   
2026-02-26  609.239990  689.299988  53.000000  55.049999  52.500000   
2026-02-27  607.289978  685.989990  53.410000  55.919998  51.430000   

Price                                                                 ...  \
Ticker             XLI         XLK        XLP        XLU         XLV  ...   
Date                                                                  ...   
2026-02-23  174.830002  138.520004  88.970001  46.680000  158.539993  ...   
2026-02-24  176.979996  140.320007  89.739998  47.200001  157.869995  ...   
2026-02-25  175.600006  143.009995  89.010002  47.360001  157.830002  ...   
2026-02-26  176.699997  141.009995  88.860001  47.180000  157.419998  ...   
2026-02-27  177.139999  138.759995  90.010002  47.730000  160.199997  ...   

Price         Volume                                                  \
Ticker           SPY         XLB         XLE         XLF         XLI   
Date                                                                   
2026-02-23  90558100  17353800.0  44013500.0  94600400.0  10566600.0   
2026-02-24  73798700  13452200.0  42921700.0  64114100.0   9756500.0   
2026-02-25  56369500  12379300.0  36207500.0  47652800.0  11757700.0   
2026-02-26  71671000  15645100.0  47033500.0  60884300.0  13764500.0   
2026-02-27  71540371  11835103.0  58897547.0  81200129.0   9676991.0   

Price                                                                   
Ticker             XLK         XLP         XLU         XLV         XLY  
Date                                                                    
2026-02-23  22021900.0  22070900.0  33130900.0  17053500.0  14604800.0  
2026-02-24  13030600.0  20364000.0  29503300.0  12887100.0  11320500.0  
2026-02-25  13296000.0  15215600.0  22356600.0  11011500.0   8776100.0  
2026-02-26  19364900.0  15925900.0  30578600.0  15344400.0   9560900.0  
2026-02-27  15251030.0  15782715.0  39718384.0  13657802.0   9208694.0  

[5 rows x 22 columns]

Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [5]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 1999-03-10 00:00:00.


Ryan Peet brought up a really good idea of using mutual funds as proxies as they have been investment vehicles for a much longer period of time. So let's see if we can build the same datapull for mutual funds, and see how far back that data goes.  
Broad Market (SPY proxy): Vanguard 500 Index (VFINX) — data back to 1976, the gold standard  
Tech (XLK): Fidelity Select Technology (FSPTX) — inception 1981  
Healthcare (XLV): Fidelity Select Health Care (FSPHX) — inception 1981  
Energy (XLE): Fidelity Select Energy (FSENX) — inception 1981  
Financials (XLF): Fidelity Select Financial Services (FIDSX) — inception 1981  
Utilities (XLU): Fidelity Select Utilities (FSUTX) — inception 1981  
*Note* Industrials is really hard because it wasn't really a sector until the late 90's early '00s. It may be best to just drop it as the only one that works is a very heavily weighted subsection of industrials  
Industrials (XLI): Fidelity Select Industrials (FCYIX) — inception 1997 (this one is shorter I actually could only get data to 2019)
Industrials2 (XLI): Fidelity Select Defense & Aerospace (FSDAX)  
Consumer Staples (XLP): Fidelity Select Consumer Staples (FDFAX) — inception 1985  
Consumer Discretionary (XLY): Fidelity Select Retailing (FSRPX) as an imperfect proxy  
Materials (XLB): Fidelity Select Materials (FSDPX) — inception 1986  
Bonds (short-term): Vanguard Short-Term Bond Index (VBISX) or use direct Treasury yields from FRED  
Bonds (long-term): Vanguard Long-Term Bond Index (VBLTX) or TLT equivalent via Barclays index data from FRED  
Money market: 3-month T-bill rate from FRED is cleaner than any fund proxy  


In [6]:
# Let's create a list of sectors
sector_mfs = ["VFINX","FSPTX","FSPHX","FSENX","FIDSX","FSUTX","FSDAX","FDFAX","FSRPX","FSDPX","VBISX","VBLTX"]
# I am going to comment out the mutual fund pull because we decided against them
#prices_mfs = yf.download(sector_mfs, period='max', auto_adjust=True)[['Close','Volume']]

In [7]:
#prices_mfs.dropna(inplace=True)
#prices_mfs.head()



In [8]:
#prices_mfs.tail()

Alright, based on this analysis, I still think that the ETF's are the strongest approach. Let's also look at some different indices that may be beneficial to our understanding of the current environment (independent variables).  

Volatility & Fear  
^VIX — CBOE Volatility Index (equity fear gauge)  
^VXN — Nasdaq volatility equivalent  
^MOVE — Bond market volatility (the "VIX for Treasuries")  

Currency  

DX-Y.NYB — DXY Dollar Index  
EURUSD=X, JPYUSD=X, CNYUSD=X — Major pairs (EUR, Yen, Yuan signal global risk appetite and trade conditions)  

Rates & Credit  

^TNX — 10-Year Treasury yield  
^TYX — 30-Year Treasury yield  
^IRX — 13-week T-Bill (short end)  
^FVX — 5-Year Treasury yield  

Commodities (macro signals)

GC=F — Gold (inflation hedge / flight to safety)  
CL=F — Crude Oil WTI (growth proxy, geopolitical risk)  
NG=F — Natural Gas  
HG=F — Copper ("Dr. Copper" — leading economic indicator)  

Credit Spreads (via ETFs since yfinance doesn't have spread data directly)  

HYG — High Yield Corporate Bonds (risk appetite)  
LQD — Investment Grade Corporate Bonds  
TLT — Long Duration Treasuries (rate sensitivity)  
SHY — Short Duration Treasuries  

Global / Geopolitical

^FTSE — UK (Brexit/European stability)  
^N225 — Nikkei (Japan / Asia Pacific) #Not working so lets try futures
NKY=F - Nikkei Futures (not spot price) Also not working so lets remove   
^HSI — Hang Seng (China exposure)  
^GSPC — S&P 500 broad market  

In [ ]:
indicies = ["^VIX","^VXN","^MOVE","DX-Y.NYB","^TNX","GC=F","CL=F","NG=F","HG=F","HYG","LQD","TLT","SHY","^FTSE","^HSI","^GSPC"] # When we removed the Credit spreads we went back to 2002, If we remove the VIX and VXN and MOVE then we can go back to late 2000.

prices_indicies = yf.download(indicies, period='max', auto_adjust=True)['Close']
# We may need to run the data with more historical data but less features and more features but less historical data to see what works best for our model but keep in mind the impact that it will have on out of sample data.

[*********************100%***********************]  9 of 9 completed


In [ ]:
#prices_indicies.dropna(inplace=True)
prices_indicies.head()

Ticker,CL=F,DX-Y.NYB,GC=F,HG=F,NG=F,^FTSE,^GSPC,^HSI,^TNX
Date,,,,,,,,,
2000-08-30,33.400002,112.139999,273.899994,0.8850,4.805,6615.100098,1502.589966,17095.880859,5.800
2000-08-31,33.099998,112.599998,278.299988,0.8850,4.780,6672.700195,1517.680054,17097.509766,5.729
2000-09-01,33.380001,111.419998,277.000000,0.8890,4.835,6795.000000,1520.770020,17333.609375,5.675
2000-09-05,33.799999,112.410004,275.799988,0.9060,4.960,6752.500000,1507.079956,17595.220703,5.683
2000-09-06,34.950001,114.120003,274.200012,0.9015,5.065,6694.700195,1492.250000,17605.230469,5.712


This amount of data only goes back to 2007, so we may want to consider dropping some of them to see if we can get data back to our origination of the macro and ETF data (around 1994)

Let's merge the data that we have here into a single dataframe and export it as a csv. 

In [38]:
print(prices.columns)
print(prices_indicies.columns)

MultiIndex([( 'Close', 'QQQ'),
            ( 'Close', 'SPY'),
            ( 'Close', 'XLB'),
            ( 'Close', 'XLE'),
            ( 'Close', 'XLF'),
            ( 'Close', 'XLI'),
            ( 'Close', 'XLK'),
            ( 'Close', 'XLP'),
            ( 'Close', 'XLU'),
            ( 'Close', 'XLV'),
            ( 'Close', 'XLY'),
            ('Volume', 'QQQ'),
            ('Volume', 'SPY'),
            ('Volume', 'XLB'),
            ('Volume', 'XLE'),
            ('Volume', 'XLF'),
            ('Volume', 'XLI'),
            ('Volume', 'XLK'),
            ('Volume', 'XLP'),
            ('Volume', 'XLU'),
            ('Volume', 'XLV'),
            ('Volume', 'XLY')],
           names=['Price', 'Ticker'])
Index(['CL=F', 'DX-Y.NYB', 'GC=F', 'HG=F', 'NG=F', '^FTSE', '^GSPC', '^HSI',
       '^TNX'],
      dtype='str', name='Ticker')


In [36]:
df_merged = pd.merge(prices, prices_indicies, on="Date", how="outer")
df_merged.head()

MergeError: Not allowed to merge between different levels. (2 levels on the left, 1 on the right)